# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [ ]:
pip install -U langchain langchain-core langchain-openai langchain-community langgraph

In [ ]:
import langchain
print(langchain.__version__)

In [ ]:
import sys
print(sys.executable)

In [ ]:
# FIXED SETUP — Compatible with LangChain v1.2+ / LangGraph
# AgentExecutor was removed in newer LangChain versions.
# We now use langgraph.prebuilt.create_react_agent instead.

import os
import tiktoken
import warnings
from dotenv import load_dotenv
from IPython.display import Markdown, display

from langchain_openai import ChatOpenAI                         # correct import path
from langchain_core.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent               # replaces AgentExecutor

warnings.filterwarnings('ignore')
load_dotenv()

# --- Tavily search tool ---
tavily_search = TavilySearchResults(api_key=os.environ.get("TAVILY_API_KEY"))

# --- LLM + token encoder ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# --- Safe Tavily wrapper (trims to 3500 tokens to stay within context limits) ---
def safe_search(query: str) -> str:
    result = tavily_search.run(query)
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)
    tokens = encoding.encode(result_text)
    return encoding.decode(tokens[:3500])

# --- Define LangChain Tool ---
tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Web search tool using Tavily. Use it to find current, real-world information."
    )
]

# --- Build agent using LangGraph (modern replacement for AgentExecutor) ---
agent = create_react_agent(model=llm, tools=tools)

# --- Helper: run agent and return the final text response ---
def run_agent(prompt: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    return result["messages"][-1].content

# Quick smoke-test
test_response = run_agent("What is LangChain in one sentence?")
print(test_response)

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [ ]:

prompt_1 = """
You are a healthcare technology analyst. Use web search to research the latest advancements
in generative AI applied to healthcare in 2025.

Return your findings as a well-structured Markdown report with the following sections:

## Generative AI in Healthcare: 2025 Advancements

### 1. Key Trends
List the 3-5 most prominent trends (e.g., diagnostic AI, drug discovery, patient care).

### 2. Notable Breakthroughs
Describe 3+ specific recent breakthroughs or product launches, with company names.

### 3. Clinical Impact
Explain how these advancements are improving patient outcomes or hospital workflows.

### 4. Challenges and Ethical Concerns
Note any regulatory, privacy, or bias issues currently being discussed.

### 5. Sources
List the URLs or publications your information comes from.

Be specific, cite real examples, and keep the tone professional.
"""

# Run it
response_1 = run_agent(prompt_1)

In [ ]:
display(Markdown(response_1))

### Exercise 2: AI Startups Landscape

Goal: Track 2025's top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [ ]:

prompt_2 = """
You are a venture capital analyst tracking the AI startup ecosystem. Use web search to
identify the top emerging AI startups of 2025 and their key innovations.

Return a clean Markdown report structured as follows:

## Top Emerging AI Startups of 2025

For each startup, provide:
- **Company Name** and founding year
- **Core Product / Innovation**: what makes them unique?
- **Funding Stage**: seed, Series A/B, etc. (if known)
- **Industry Focus**: e.g., healthcare, fintech, robotics

Cover at least 6 startups across different sectors.

### Honorable Mentions
Include 2-3 other noteworthy startups with a one-line description each.

### Trends to Watch
Summarize 2-3 overarching trends visible across these startups.

### Sources
List the URLs or publications referenced.

Use real, verifiable company names and products only.
"""

# Run it
response_2 = run_agent(prompt_2)

In [ ]:
display(Markdown(response_2))

### Exercise 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI's ChatGPT Team and Anthropic's Claude Pro.
- Ensure the agent outputs a structured response in Markdown.


In [ ]:

prompt_3 = """
You are a technology product reviewer. Use web search to compare OpenAI's ChatGPT Team plan
and Anthropic's Claude Pro plan as of 2025.

Produce a structured Markdown comparison report:

## ChatGPT Team vs Claude Pro: 2025 Comparison

### Pricing
Provide the monthly and annual pricing for both plans, including any per-seat costs.

### Feature Comparison Table
Use a Markdown table with rows for: context window, file uploads, web browsing,
image generation, API access, team collaboration features, and data privacy policies.

### Performance and Model Quality
Summarize how each model performs based on benchmarks or recent user reviews.

### Best For
Describe the ideal user or team type for each product.

### Verdict
Give a balanced, unbiased recommendation based on use case.

### Sources
List the URLs referenced for pricing and feature information.

Use real, up-to-date pricing and features from official websites where possible.
"""

# Run it
response_3 = run_agent(prompt_3)

In [ ]:
display(Markdown(response_3))

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.

**Use Case chosen: Real-Time Job Market Intelligence for Data Scientists**

A recruiter or job seeker could use this agent to get an instant, structured snapshot of the current data science job market — covering in-demand skills, salary ranges, top hiring companies, and remote work trends — saving hours of manual research.